In [ ]:
import json
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO 
import os

pd.io.html._remove_whitespace = lambda v: v.replace('\xa0', ' ').replace('Â', ' ')

url = "https://dev.twitch.tv/docs/eventsub/eventsub-reference/"
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

In [ ]:
def get_tables():
    ignore_tables = ("Drop Entitlement Grant Event", "Request fields", "Response fields")
    for section in soup.find_all("section"):
        if not section.find("h1", id="objects"):
            continue
            
        header = None
        for element in section.descendants:
            if element.name == "h2" or element.name == "h3":
                header = element.get_text()
            if element.name == "table":
                if header in ignore_tables:
                    continue
                    
                table = pd.read_html(StringIO(str(element)))[0]
                table = table.rename(columns={'Field': 'Name', 'Param': 'Name'})
                table['Name'] = table['Name'].str.replace('Â', ' ')
                table['Type'] = table['Type'].str.lower()
                yield (header, table)
                header = None
                table = None

# Intermediate Representation

parse tables into a tree structure which encodes labels, types, and hierarchy.
this representation can then be used to convert to a target representation i.e. OpenAPI schema

In [ ]:
class Node:
    def __init__(self, parent, node_type : str, label : str, depth : int, required : bool = True, docs : str = ""):
        self.parent = parent
        self.children = []
        
        self.node_type = node_type.lower()
        self.key = label
        self.label = label
        
        self.docs = docs
        self.depth = depth

    def add_child(self, node_type, node_label, docs = "", lvl=0):
        # this label is known to be encoded as a string in the source
        if (node_label == "charity_donation"):
            node_type = "object"

        # this label is problematic, is a keyword in the target representation
        if node_label == 'delete':
            node_label = 'delete_'
            
        node = Node(self, node_type, node_label, docs, lvl)

        make_singular = False
        match node.node_type.lower():
            case 'object[]' | '[]object':
                make_singular = True
            case "array":
                make_singular = True
            case s if s.endswith('s'):
                make_singular = True

        if make_singular:
            if node.label.endswith('ies'):
                node.label = node.label[:-3] + 'y'
            elif node.label.endswith('s'):
                node.label = node.label[:-1]
            elif node.label == 'terms_found':
                node.label = 'term_found'
            else:
                print(f"how to handle making {node.label} singular?")
        
        self.children.append(node)
        return node

    def __eq__(self, other):
        return self.identity() == other.identity()

    def __hash__(self):
        return self.identity().__hash__()

    def identity(self):
        identity = f"{self.depth * '  '}{self.label} -> {self.node_type}\n"
        for child in self.children:
            identity += child.identity()

        return identity
        
    def print(self):
        print(self.identity())

def create_tree(items, name):        
    root = Node(None, "object", name, 0)
    prev_node = None

    for label, node_type, _ in items:
        stripped_label = label.lstrip()
        indent = (len(label) - len(stripped_label))

        if (stripped_label == "community_gift_id" and indent == 5):
            indent=6

        if not prev_node:
            prev_node = root.add_child(node_type, stripped_label, indent)
            continue

        # we are a sibling!
        if indent == prev_node.depth:
            # error in data, siblings cannot have the same keys
            # this occurrs a couple of times, they are (currently) 
            # always sequential
            if prev_node.label == stripped_label:
                continue
            
            prev_node = prev_node.parent.add_child(node_type, stripped_label, indent)
            continue

        # we are prev_nodes child
        if indent > prev_node.depth:
            prev_node = prev_node.add_child(node_type, stripped_label, indent)
            continue

        while indent < prev_node.depth:
            prev_node = prev_node.parent
            
        prev_node = prev_node.parent.add_child(node_type, stripped_label, indent)
       
    return root

# Create C++ Represenation

In [ ]:
def create_struct(node, duplicates):
    root = node
    while root.parent:
        root = root.parent
        
    def get_class_name(name):
        if name in duplicates:
            name = root.label + '_' + name
        return name.title().replace('_', '')
        
    children = []
    class_name = get_class_name(node.label)
    struct_str = f"class {class_name} {{\npublic:\n"
    struct_str += f"  {class_name}(){{}}\n\n"
    struct_str += f"  {class_name}(boost::json::object obj) {{\n"

    for child in node.children:
        match child.node_type:
            case s if (s.endswith('s') or s == "array" or s == "object[]"):
                struct_str += f'    if (!obj["{child.key}"].is_null()) {{\n'
                struct_str += f'      for (auto item : obj.at("{child.key}").as_array()) {{\n'
                struct_str += f'          {child.label}.push_back(item.as_object());\n'
                struct_str += f'      }}\n'
                struct_str += f'    }}\n\n'
            case "int" | "int (or null)" | "integer":
                struct_str += f'    {child.label} = value_as_int(obj, "{child.key}");\n'

            case "boolean" | "bool":
                struct_str += f'    {child.label} = obj.at("{child.key}").as_bool();\n'

            case "string":
                struct_str += f'    {child.label} = value_as_string(obj, "{child.label}");\n'

            case "string[]" | "[]string":
                struct_str += f'    if (!obj["{child.key}"].is_null()) {{\n'
                struct_str += f'      for (auto item : obj.at("{child.key}").as_array()) {{\n'
                struct_str += f'          {child.label}.push_back(item.as_string().c_str());\n'
                struct_str += f'      }}\n'
                struct_str += f'    }}\n\n'
            case _:
                child_class_name = get_class_name(child.label)
                struct_str += f'    if (!obj["{child.key}"].is_null()) {{\n'
                struct_str += f'      {child.label} = {child_class_name}(obj.at("{child.key}").as_object());\n'
                struct_str += f'    }}\n\n'
     
                
    struct_str += f"  }}\n\n"
        
    for child in node.children:
        match child.node_type:
            case s if (s.endswith('s') or s == "array" or s == "object[]"):
                child_class_name = get_class_name(child.label)
                thing, thing_children = create_struct(child, duplicates)
                if thing_children:
                    children.extend(thing_children)
                if thing:
                    children.append(thing)

                struct_str += f"  std::vector<{child_class_name}> {child.label};\n"

            case "int" | "int (or null)" | "integer":
                struct_str += f"  int {child.label};\n"
            case "boolean" | "bool":
                struct_str += f"  bool {child.label};\n"
            case "string":
                struct_str += f"  std::string {child.label};\n"
            case "string[]" | "[]string":
                struct_str += f"  std::vector<std::string> {child.label};\n"
            case _:
                child_class_name = get_class_name(child.label)
                thing, thing_children = create_struct(child, duplicates)
                if thing_children:
                    children.extend(thing_children)
                if thing:
                    children.append(thing)
                struct_str += f"  std::optional<{child_class_name}> {child.label};\n"    


    struct_str += "};"

    return struct_str, children

In [ ]:
# intermediate representation construction
trees = dict()
for header, table in get_tables():
    items = [(item[0], item[1], item[2]) for item in table.itertuples(index=False)]
    try:
        tree = create_tree(items, header.replace('-', '').replace(' ', '_').lower())
        trees[tree.label]=tree
    except:
        print(header)

# de-duplication step
seen = set()
duplicates = set()
base_types = set(["int", "int (or null)", "integer", "boolean", "bool", "string", "string[]", "[]string"])
def dfs(t):
    if t.node_type in base_types:
        return
        
    if t.label.strip() in seen:
            duplicates.add(t.label.strip())
    else:   
        seen.add(t.label.strip())

    for child in t.children:
        dfs(child)

for key, tree in trees.items():
    dfs(tree)

# target representation construction
objects = []
for key, tree in trees.items():  
    struct, children = create_struct(tree, duplicates)
    children = list(dict.fromkeys(children))
    objects.append((struct, children))
    
output_header = """#pragma once

#include <boost/json/object.hpp>
#include <string>
#include <vector>

#include <boost/json.hpp>

std::string value_as_string(boost::json::object obj, const std::string name) {
  if (obj[name].is_null())
    return "";

  return obj.at(name).as_string().c_str();
}

int value_as_int(boost::json::object obj, const std::string name) {
  if (obj[name].is_null())
    return -1;

  return obj.at(name).as_int64();
}

"""

# create final file
with open("out/messages.hpp", "w") as f:
    f.write(output_header)
    
    for item, children in objects:
        if item is None or children is None:
            continue 
        for child in children:
            f.write(child)
            f.write('\n\n')
        f.write(item)
        f.write('\n\n')